In [20]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nalisha/job-salary-prediction-dataset/job_salary_prediction_dataset.csv


In [21]:
import pandas as pd
import numpy as np 
import seaborn as sns

In [22]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/nalisha/job-salary-prediction-dataset/job_salary_prediction_dataset.csv", encoding='latin1')
df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,AI Engineer,10,Bachelor,2,Healthcare,Medium,India,Hybrid,2,109413
1,Data Analyst,5,Bachelor,17,Telecom,Small,Australia,No,0,93764
2,Frontend Developer,18,PhD,4,Media,Medium,Singapore,No,1,148123
3,Business Analyst,19,PhD,13,Retail,Medium,Canada,Yes,0,189123
4,Product Manager,15,Bachelor,7,Manufacturing,Large,Sweden,Yes,0,165069


In [23]:
#cheak null value
df.shape

(250000, 10)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 10 columns):
 #   Column            Non-Null Count   Dtype 
---  ------            --------------   ----- 
 0   job_title         250000 non-null  object
 1   experience_years  250000 non-null  int64 
 2   education_level   250000 non-null  object
 3   skills_count      250000 non-null  int64 
 4   industry          250000 non-null  object
 5   company_size      250000 non-null  object
 6   location          250000 non-null  object
 7   remote_work       250000 non-null  object
 8   certifications    250000 non-null  int64 
 9   salary            250000 non-null  int64 
dtypes: int64(4), object(6)
memory usage: 19.1+ MB


In [25]:
df.isnull().sum()

job_title           0
experience_years    0
education_level     0
skills_count        0
industry            0
company_size        0
location            0
remote_work         0
certifications      0
salary              0
dtype: int64

In [26]:
#featur engineering
#how much salary in domain as experince based
salary_exp = df.groupby(['job_title', 'experience_years'])['salary'].mean().reset_index()
salary_exp.head()

,job_title,experience_years,salary
0,AI Engineer,0,147693.022495
1,AI Engineer,1,146833.154536
2,AI Engineer,2,150766.032661
3,AI Engineer,3,154537.218496
4,AI Engineer,4,158308.678606


In [27]:
#df = df.merge(salary_exp, on=['job_title', 'experience_years'], how='left')
#df.rename(columns={'salary_y': 'avg_salary_by_role_exp'}, inplace=True)
#df.head()

In [28]:
#how much salary got to which setor in domain as experince based
industry = df.groupby(['industry', 'job_title', 'experience_years'])['salary'].median().reset_index()
industry.head()

,industry,job_title,experience_years,salary
0,Consulting,AI Engineer,0,148002.0
1,Consulting,AI Engineer,1,141120.0
2,Consulting,AI Engineer,2,145567.0
3,Consulting,AI Engineer,3,148137.0
4,Consulting,AI Engineer,4,153764.5


In [29]:
#how much salary got to which location in domain as experince based
location = df.groupby(['location', 'job_title', 'experience_years'])['salary'].median().reset_index()
location.head()

,location,job_title,experience_years,salary
0,Australia,AI Engineer,0,139778.5
1,Australia,AI Engineer,1,137980.0
2,Australia,AI Engineer,2,141333.0
3,Australia,AI Engineer,3,144260.0
4,Australia,AI Engineer,4,154750.0


In [32]:
from sklearn.preprocessing import LabelEncoder

categorical_columns = df.select_dtypes(include=['object']).columns

for col in categorical_columns:
    le = LabelEncoder()   # har column ke liye new encoder
    df[col] = le.fit_transform(df[col].astype(str))

df.head()

,job_title,experience_years,education_level,skills_count,industry,company_size,location,remote_work,certifications,salary
0,0,10,0,2,4,2,3,0,2,109413
1,5,5,0,17,9,3,0,1,0,93764
2,8,18,4,4,6,2,6,1,1,148123
3,2,19,4,13,7,2,1,2,0,189123
4,10,15,0,7,5,1,7,2,0,165069


In [39]:
# warnings ignore
import warnings
warnings.filterwarnings('ignore')

# models
from xgboost import XGBRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from lightgbm import LGBMRegressor

# model selection
from sklearn.model_selection import train_test_split, KFold, cross_val_score, GridSearchCV

# metrics
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

# explainability
import lime
import lime.lime_tabular

In [40]:
#train model
X = df.drop('salary', axis=1)   # features
y = df['salary']                # target

In [44]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE
)

In [45]:
RANDOM_STATE = 42
models={
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(),
    "Lasso Regression": Lasso(),
    "Decision Tree": DecisionTreeRegressor(random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(random_state=RANDOM_STATE),
    "XGBoost": XGBRegressor(random_state=RANDOM_STATE),
    "LightGBM": LGBMRegressor(random_state=RANDOM_STATE, verbose=-1),
    "KNN Regressor": KNeighborsRegressor()
}

In [46]:
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

all_model_scores = {}

for model_name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=kf, scoring="r2")

    mean_r2 = np.mean(scores)
    std_r2 = np.std(scores)
    
    print(f"{model_name} - R2 Score: {mean_r2:.4f} (+/- {std_r2:.4f})")
    all_model_scores[model_name] = {
        'fold_scores': scores,
        'mean_score': round(mean_r2, 2)
    }

Linear Regression - R2 Score: 0.4594 (+/- 0.0013)
Ridge Regression - R2 Score: 0.4594 (+/- 0.0013)
Lasso Regression - R2 Score: 0.4594 (+/- 0.0013)
Decision Tree - R2 Score: 0.9264 (+/- 0.0010)
Random Forest - R2 Score: 0.9674 (+/- 0.0003)
XGBoost - R2 Score: 0.9778 (+/- 0.0003)
LightGBM - R2 Score: 0.9773 (+/- 0.0002)
KNN Regressor - R2 Score: 0.6950 (+/- 0.0014)
